# Analiza streszczeń litaratury młodzieżowej

## Import bibliotek


In [ ]:
import os
import json
import string
import morfeusz2
import numpy as np
import pandas as pd
import operator as op
import itertools as it
from wordcloud import WordCloud
from matplotlib import pyplot as plt
from nltk import ngrams
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from scipy.cluster import hierarchy
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
import nltk
nltk.download('punkt_tab')
if "." not in nltk.data.path:
    nltk.data.path.append(".")

# Zdefiniowanie funkcji pomocniczych

In [ ]:
def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))

In [ ]:
def create_dir(path):
    if not os.path.exists(path):
        os.mkdir(path)

# Przygotowanie danych wejściowych

## Utworzenie korpusu dokumentów

In [ ]:
corpus_dir = "./literature"
corpus = PlaintextCorpusReader(corpus_dir, r'.*\.txt')
files_names = corpus.fileids()
titles = [file_name.removesuffix(".")[0] for file_name in files_names]

In [ ]:
documents_dict = {title: corpus.raw(file_name) for title, file_name in zip(titles, files_names)}
print_dict(documents_dict)

In [ ]:
documents_params = {}
for title, document in documents_dict.items():
    documents_params[title] = {
        "pre": len(word_tokenize(document, language="polish")),
    }
print_dict(documents_params)

### Utworzenie macierzy częstości (przed wstępnym przetwarzaniem)

In [ ]:
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df_pre.columns = ["content"]
display(documents_df_pre)

In [ ]:
cv = CountVectorizer()
tv = TfidfVectorizer()

In [ ]:
matrix_tf = cv.fit_transform(documents_df_pre["content"])
sparsity_tf = 1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))

In [ ]:
matrices_params = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparsity": sparsity_tf,
    }
}
print_dict(matrices_params)

### Wstępne przygotowanie dokumentów

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords_file.close()
stopwords = stopwords[4:]

In [ ]:
def lemmatize(text):
    lemmatizer = morfeusz2.Morfeusz()
    segments = it.groupby(lemmatizer.analyse(text), op.itemgetter(0,1))
    def disambiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
        _, lemma = min(pairs)
        return lemma.split(":")[0]
    return " ".join(filter(str.isalpha, (disambiguate(group) for _, group in segments)))


In [ ]:
for title, document in documents_dict.items():
    documents_dict[title] = documents_dict[title].lower()
    documents_dict[title] = "".join(filter(lambda char: char not in string.punctuation, documents_dict[title]))
    documents_dict[title] = lemmatize(documents_dict[title])
    documents_dict[title] = " ".join(filter(lambda word: word not in stopwords, word_tokenize(documents_dict[title], language="polish")))
print_dict(documents_dict)

In [ ]:
for title, document in documents_dict.items():
    documents_params[title]["post"] = len(word_tokenize(document, language="polish"))
print_dict(documents_params)

In [ ]:
documents_params_df = pd.DataFrame.from_dict(documents_params, orient="index")
documents_params_df.columns = ["pre", "post"]
documents_params_df["change"] = documents_params_df["post"] - documents_params_df["pre"]
documents_params_df["change_percent"] = documents_params_df["change"] / documents_params_df["pre"] * 100
display(documents_params_df)

# Utworzenie macierzy częstości (po wstępnym przetwarzaniu)

In [ ]:
documents_df_post = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df_post.columns = ["content"]
display(documents_df_post)

In [ ]:
matrix_tf = cv.fit_transform(documents_df_post["content"])
matrix_tfidf = tv.fit_transform(documents_df_post["content"])
sparsity_tf = 1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))

In [ ]:
matrices_params["post"] = {
    "words": matrix_tf.shape[1],
    "sparsity": sparsity_tf,
}
print_dict(matrices_params)

In [ ]:
matrices_params_df = pd.DataFrame.from_dict(matrices_params, orient="index")
display(matrices_params_df)